In [ ]:
# Cell 1: Import libraries and configure dependencies
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model
import tensorflow as tf
import os
from pathlib import Path
import yaml

# Always resolve path relative to the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
model_path = os.path.join(project_root, "models", "rml2016", "rml2016_lstm_rnn_2024.keras")

print("Resolved model path:", model_path)
assert os.path.exists(model_path), f"Model file not found: {model_path}"

model = load_model(model_path)
print("Model loaded successfully.")

# Resolve dataset path from local config (written by 10_download_data.ipynb)
cfg_path = Path(project_root) / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    local_cfg = yaml.safe_load(cfg_path.read_text())
    rml2016_pkl = local_cfg.get('datasets', {}).get('rml2016', {}).get('pkl', '/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl')
else:
    rml2016_pkl = '/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl'

print('Resolved RML2016 data path:', rml2016_pkl)
assert os.path.exists(rml2016_pkl), f'Dataset not found: {rml2016_pkl}'

# Load the data
with open(rml2016_pkl, "rb") as f:
    data = pickle.load(f, encoding="latin1")

# Prepare the data using your specific format
def prepare_data(data):
    X, y = [], []

    for (mod_type, snr), signals in data.items():
        for signal in signals:
            iq_signal = np.vstack([signal[0], signal[1]]).T
            snr_signal = np.full((128, 1), snr)
            combined_signal = np.hstack([iq_signal, snr_signal])
            X.append(combined_signal)
            y.append(mod_type)

    X = np.array(X)
    y = np.array(y)
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42
    )
    X_train = X_train.reshape(-1, X_train.shape[1], X_train.shape[2])
    X_test = X_test.reshape(-1, X_test.shape[1], X_test.shape[2])

    return X_train, X_test, y_train, y_test, label_encoder

# Prepare the data
X_train, X_test, y_train, y_test, label_encoder = prepare_data(data)

# Make predictions on the test set
y_pred = np.argmax(model.predict(X_test, verbose=False), axis=1)

# Plot the confusion matrix for all SNR levels
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", 
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix for Modulation Classification (All SNR Levels)")
plt.show()

# Print the classification report
print("Classification Report for Modulation Types:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Plot feature importance (if using a tree-based model)
# Plot confusion matrix for SNR > 5 dB subset
snr_above_5_indices = np.where(X_test[:, 0, 2] > 5)  # Assuming SNR values are in the third column
X_test_snr_above_5 = X_test[snr_above_5_indices]
y_test_snr_above_5 = y_test[snr_above_5_indices]

# Make predictions on the SNR > 5 dB subset
y_pred_snr_above_5 = np.argmax(model.predict(X_test_snr_above_5, verbose=False), axis=1)

# Plot confusion matrix for SNR > 5 dB
conf_matrix_snr_above_5 = confusion_matrix(y_test_snr_above_5, y_pred_snr_above_5)
plt.figure(figsize=(12, 10))
sns.heatmap(conf_matrix_snr_above_5, annot=True, fmt="d", cmap="Blues", 
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix for Modulation Classification (SNR > 5 dB)")
plt.show()

# Print the classification report for SNR > 5 dB
print("Classification Report for Modulation Types (SNR > 5 dB):")
print(classification_report(y_test_snr_above_5, y_pred_snr_above_5, target_names=label_encoder.classes_))


In [ ]:
# Cell 2: Import libraries and configure dependencies
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Load data and prepare using your prepare_data function
X_train, X_test, y_train, y_test, label_encoder = prepare_data(data)

# Evaluate accuracy for each SNR level
unique_snrs = sorted(set(X_test[:, 0, -1]))  # SNR values are in the last column of each sample
accuracy_per_snr = []

for snr in unique_snrs:
    # Select samples with the current SNR
    snr_indices = np.where(X_test[:, 0, -1] == snr)
    X_snr = X_test[snr_indices]
    y_snr = y_test[snr_indices]

    # Predict and calculate accuracy
    y_pred = np.argmax(model.predict(X_snr,verbose=0), axis=1)
    accuracy = accuracy_score(y_snr, y_pred)
    accuracy_per_snr.append(accuracy * 100)  # Convert to percentage

    print(f"SNR: {snr} dB, Accuracy: {accuracy * 100:.2f}%")

# Find the peak accuracy and its corresponding SNR
peak_accuracy = max(accuracy_per_snr)
peak_snr = unique_snrs[accuracy_per_snr.index(peak_accuracy)]

# Plot Recognition Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(unique_snrs, accuracy_per_snr, 'b-o', label='Recognition Accuracy')
plt.xlabel("SNR (dB)")
plt.ylabel("Recognition Accuracy (%)")
plt.title(f"Recognition Accuracy vs. SNR (Peak Accuracy: {peak_accuracy:.2f}%)")

# Mark the peak accuracy point
plt.plot(peak_snr, peak_accuracy, 'ro')  # Red dot at the peak
plt.text(peak_snr, peak_accuracy + 1, f"{peak_accuracy:.2f}%", 
         ha='center', va='bottom', fontsize=10, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

plt.legend()
plt.grid(True)
plt.ylim(0, 100)
plt.show()


In [ ]:
# Cell 3: Import libraries and configure dependencies
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import tensorflow as tf
# Set TensorFlow logging level to suppress most of the output
tf.get_logger().setLevel('ERROR')
import numpy as np

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

def rnn_lstm_plot_accuracy_v_snr_per_classification(model, X_test, y_test, label_encoder):
    # Get unique modulation types using label encoder classes
    unique_modulations = label_encoder.classes_  # This will be the actual class names
    unique_snrs = sorted(set(X_test[:, 0, -1]))  # Get unique SNR levels (assuming SNR is in the last column of each sample)

    # Initialize a list to store modulation, accuracy per SNR, and peak accuracy for sorting
    modulation_traces = []

    # Calculate accuracy for each modulation type and SNR level
    for mod_index, mod in enumerate(unique_modulations):
        accuracies = []
        for snr in unique_snrs:
            # Select samples with the current modulation type and SNR
            mod_snr_indices = np.where((y_test == mod_index) & (X_test[:, 0, -1] == snr))
            X_mod_snr = X_test[mod_snr_indices]
            y_mod_snr = y_test[mod_snr_indices]

            # Predict and calculate accuracy
            if len(y_mod_snr) > 0:  # Check if there are samples for this SNR and modulation type
                y_pred = np.argmax(model.predict(X_mod_snr, verbose=False), axis=1)
                accuracy = accuracy_score(y_mod_snr, y_pred) * 100  # Convert to percentage
            else:
                accuracy = np.nan  # No data for this SNR-modulation combination

            accuracies.append(accuracy)

        # Calculate peak accuracy for this modulation type
        valid_accuracies = [acc for acc in accuracies if not np.isnan(acc)]
        peak_accuracy = max(valid_accuracies) if valid_accuracies else 0
        peak_snr = unique_snrs[accuracies.index(peak_accuracy)] if peak_accuracy > 0 else None

        # Store the modulation trace data along with peak accuracy for sorting
        modulation_traces.append((mod, accuracies, peak_accuracy, peak_snr))

    # Sort the modulation types by peak accuracy in descending order
    modulation_traces = sorted(modulation_traces, key=lambda x: x[2], reverse=True)

    # Plot Recognition Accuracy vs. SNR for each modulation type (All)
    plt.figure(figsize=(12, 8))
    for mod, accuracies, peak_accuracy, peak_snr in modulation_traces:
        label = f'{mod} (Peak: {peak_accuracy:.2f}% at {peak_snr} dB)' if peak_accuracy > 0 else mod
        plt.plot(unique_snrs, accuracies, '-o', label=label)

        # Mark the peak accuracy point if it exists
        if peak_accuracy > 0 and peak_snr is not None:
            plt.plot(peak_snr, peak_accuracy, 'ro')  # Red dot at the peak
            plt.text(peak_snr, peak_accuracy + 1, f"{peak_accuracy:.2f}%", 
                    ha='center', va='bottom', fontsize=10, 
                    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

    plt.xlabel("SNR (dB)")
    plt.ylabel("Recognition Accuracy (%)")
    plt.title("Recognition Accuracy vs. SNR per Modulation Type (All Classifications)")
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.ylim(0, 110)
    plt.xlim(min(unique_snrs), max(unique_snrs))
    plt.show()

    # Plot Recognition Accuracy vs. SNR for each modulation type (Top 6 by Peak Accuracy)
    plt.figure(figsize=(12, 8))
    for mod, accuracies, peak_accuracy, peak_snr in modulation_traces[:6]:
        label = f'{mod} (Peak: {peak_accuracy:.2f}% at {peak_snr} dB)' if peak_accuracy > 0 else mod
        plt.plot(unique_snrs, accuracies, '-o', label=label)

        # Mark the peak accuracy point if it exists
        if peak_accuracy > 0 and peak_snr is not None:
            plt.plot(peak_snr, peak_accuracy, 'ro')  # Red dot at the peak
            plt.text(peak_snr, peak_accuracy + 1, f"{peak_accuracy:.2f}%", 
                    ha='center', va='bottom', fontsize=10, 
                    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

    plt.xlabel("SNR (dB)")
    plt.ylabel("Recognition Accuracy (%)")
    plt.title("Recognition Accuracy vs. SNR per Modulation Type (Top 6 by Peak Accuracy)")
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.ylim(0, 110)
    plt.xlim(min(unique_snrs), max(unique_snrs))
    plt.show()

    # Plot Recognition Accuracy vs. SNR for each modulation type (Bottom 5 by Peak Accuracy)
    plt.figure(figsize=(12, 8))
    for mod, accuracies, peak_accuracy, peak_snr in modulation_traces[-5:]:
        label = f'{mod} (Peak: {peak_accuracy:.2f}% at {peak_snr} dB)' if peak_accuracy > 0 else mod
        plt.plot(unique_snrs, accuracies, '-o', label=label)

        # Mark the peak accuracy point if it exists
        if peak_accuracy > 0 and peak_snr is not None:
            plt.plot(peak_snr, peak_accuracy, 'ro')  # Red dot at the peak
            plt.text(peak_snr, peak_accuracy + 1, f"{peak_accuracy:.2f}%", 
                    ha='center', va='bottom', fontsize=10, 
                    bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))

    plt.xlabel("SNR (dB)")
    plt.ylabel("Recognition Accuracy (%)")
    plt.title("Recognition Accuracy vs. SNR per Modulation Type (Bottom 5 by Peak Accuracy)")
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.ylim(0, 110)
    plt.xlim(min(unique_snrs), max(unique_snrs))
    plt.show()

    
rnn_lstm_plot_accuracy_v_snr_per_classification(model, X_test, y_test, label_encoder)
